# Prédiction des maladies cardiaques : SVM, kNN et réseau de neurones

Notebook de l'article [Prédiction des maladies cardiaques par Machine Learning](https://moncoachdata.com/blog/prediction-des-maladies-cardiaques/) (MonCoachData).

Données : *South African Heart Disease* (462 hommes d'une région à haut risque, à l'ouest du Cap), [fiche OpenML](https://www.openml.org/d/1498). Le fichier `SAheart.csv` est hébergé dans le dépôt GitHub de MonCoachData et lu directement depuis son adresse : rien à télécharger.

Exercice pédagogique, pas un outil de diagnostic médical.

## 1. Charger et vérifier les données

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

In [ ]:
URL = 'https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/prediction-des-maladies-cardiaques/SAheart.csv'
data = pd.read_csv(URL, index_col='row.names')

data.head()

In [ ]:
print(data.shape)
print('Valeurs manquantes :', data.isnull().sum().sum())
print(data['chd'].value_counts(normalize=True).round(3))

`famhist` est du texte (Present ou Absent) : on le transforme en 0 et 1. `chd` est déjà codé en 0 (témoin) et 1 (cas).

In [ ]:
# famhist : 'Present' devient 1, 'Absent' devient 0
data['famhist'] = (data['famhist'] == 'Present').astype(int)

data.describe().T.round(2)

## 2. Visualisation : cas et témoins

In [ ]:
data.groupby('chd').mean().round(2).T

In [ ]:
colonnes_numeriques = ['sbp', 'tobacco', 'ldl', 'adiposity', 'typea', 'obesity', 'alcohol', 'age']

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, colonne in zip(axes.ravel(), colonnes_numeriques):
    sns.boxplot(data=data, x='chd', y=colonne, ax=ax)
    ax.set_xlabel('chd (0 = témoin, 1 = cas)')
plt.tight_layout()
plt.show()

## 3. Le rôle de l'âge

In [ ]:
tranches = pd.cut(data['age'], [14, 29, 39, 49, 64], labels=['15-29', '30-39', '40-49', '50-64'])

data.groupby(tranches, observed=True).agg(
    effectif=('age', 'size'),
    tabac=('tobacco', 'mean'),
    alcool=('alcohol', 'mean'),
    obesite=('obesity', 'mean'),
    part_de_cas=('chd', 'mean'),
).round(2)

In [ ]:
groupe = data['chd'].map({0: 'témoin', 1: 'cas'})

sns.scatterplot(data=data, x='age', y='obesity', hue=groupe, alpha=0.7)
plt.xlabel('Âge (ans)')
plt.ylabel('Obésité')
plt.legend(title='Groupe')
plt.show()

In [ ]:
sns.scatterplot(data=data, x='age', y='tobacco', hue=groupe, alpha=0.7)
plt.xlabel('Âge (ans)')
plt.ylabel('Consommation cumulée de tabac (kg)')
plt.legend(title='Groupe')
plt.show()

In [ ]:
sns.scatterplot(data=data, x='age', y='alcohol', hue=groupe, alpha=0.7)
plt.xlabel('Âge (ans)')
plt.ylabel('Consommation d\'alcool')
plt.legend(title='Groupe')
plt.show()

## 4. Séparer les données et fixer une référence

Les variables sont mises à l'échelle **après** la séparation entraînement/test (dans un `Pipeline`), pour que le test ne « fuite » pas dans l'entraînement. La référence est un « modèle » qui répond toujours « pas de maladie ».

In [ ]:
from sklearn.model_selection import train_test_split

colonnes = ['sbp', 'tobacco', 'ldl', 'adiposity', 'famhist', 'typea', 'obesity', 'alcohol', 'age']
X = data[colonnes]
y = data['chd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)
print(len(X_train), len(X_test))

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

def evaluer(nom, y_vrai, y_pred):
    print(nom)
    print(confusion_matrix(y_vrai, y_pred))
    print(f"Exactitude : {accuracy_score(y_vrai, y_pred):.3f}")
    print(f"Précision  : {precision_score(y_vrai, y_pred, zero_division=0):.3f}")
    print(f"Rappel     : {recall_score(y_vrai, y_pred):.3f}")

# Référence : un « modèle » qui répond toujours 0 (pas de maladie)
reference = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
evaluer('Référence (toujours 0)', y_test, reference.predict(X_test))

## 5. Modèle 1 : SVM

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_clf = make_pipeline(StandardScaler(), SVC(kernel='linear'))
svm_clf.fit(X_train, y_train)

y_pred_svm = svm_clf.predict(X_test)
evaluer('SVM', y_test, y_pred_svm)

## 6. Modèle 2 : kNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_clf = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_clf.fit(X_train, y_train)

y_pred_knn = knn_clf.predict(X_test)
evaluer('kNN', y_test, y_pred_knn)

In [ ]:
# Le même kNN, mais sans mise à l'échelle
knn_brut = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
accuracy_score(y_test, knn_brut.predict(X_test))

## 7. Modèle 3 : réseau de neurones (Keras)

Le résultat peut varier de quelques points selon la version de TensorFlow et la machine.

In [ ]:
import keras
from keras import layers

def construire_ann():
    modele = keras.Sequential([
        keras.Input(shape=(9,)),
        layers.Dense(6, activation='relu'),   # première couche cachée
        layers.Dense(6, activation='relu'),   # seconde couche cachée
        layers.Dense(1, activation='sigmoid'),  # sortie : probabilité d'être un cas
    ])
    modele.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return modele

keras.utils.set_random_seed(1234)

# Un réseau de neurones a aussi besoin de données mises à l'échelle
scaler = StandardScaler().fit(X_train)

ann = construire_ann()
ann.fit(scaler.transform(X_train), y_train, batch_size=10, epochs=100, verbose=0)

y_pred_ann = (ann.predict(scaler.transform(X_test), verbose=0) > 0.5).astype(int).ravel()
evaluer('ANN', y_test, y_pred_ann)

## 8. Comparer les trois approches : validation croisée à 10 plis

Un seul découpage de 93 hommes est fragile : chaque homme pèse environ 1 point d'exactitude. Avec la validation croisée, chaque homme est testé une fois. La cellule suivante prend quelques minutes (10 entraînements du réseau de neurones).

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=1234)
metriques = {'accuracy': 'accuracy',
             'precision': make_scorer(precision_score, zero_division=0),
             'recall': 'recall'}

scores = {}
for nom, modele in [('Référence', DummyClassifier(strategy='most_frequent')),
                    ('SVM', svm_clf), ('kNN', knn_clf)]:
    resultat = cross_validate(modele, X, y, cv=cv, scoring=metriques)
    scores[nom] = {m: resultat['test_' + m] for m in metriques}

# Le réseau de neurones : une boucle, car il faut refaire la mise à l'échelle à chaque pli
scores['ANN'] = {m: [] for m in metriques}
for idx_train, idx_test in cv.split(X, y):
    keras.backend.clear_session()   # libère la mémoire entre deux plis
    sc = StandardScaler().fit(X.iloc[idx_train])
    reseau = construire_ann()
    reseau.fit(sc.transform(X.iloc[idx_train]), y.iloc[idx_train], batch_size=10, epochs=100, verbose=0)
    pred = (np.asarray(reseau(sc.transform(X.iloc[idx_test]))) > 0.5).astype(int).ravel()
    scores['ANN']['accuracy'].append(accuracy_score(y.iloc[idx_test], pred))
    scores['ANN']['precision'].append(precision_score(y.iloc[idx_test], pred, zero_division=0))
    scores['ANN']['recall'].append(recall_score(y.iloc[idx_test], pred))

In [ ]:
moyennes = pd.DataFrame({nom: {m: np.mean(v) for m, v in s.items()} for nom, s in scores.items()}).T
ecarts = pd.DataFrame({nom: {m: np.std(v) for m, v in s.items()} for nom, s in scores.items()}).T

moyennes.columns = ['Exactitude', 'Précision', 'Rappel']
(moyennes * 100).round(1)

In [ ]:
(ecarts * 100).round(1).rename(columns={'accuracy': 'Exactitude', 'precision': 'Précision', 'recall': 'Rappel'})

In [ ]:
ax = (moyennes.loc[['SVM', 'kNN', 'ANN']] * 100).plot(kind='bar', figsize=(6, 6), ylim=[0, 100], rot=0)
ax.set_ylabel('%')
ax.set_title('Comparaison des 3 modèles (validation croisée à 10 plis)')
ax.grid(axis='x', visible=False)
plt.show()

## 9. À toi de jouer

Change le `random_state` du découpage et regarde à quel point les résultats d'un seul découpage bougent. Essaie aussi une régression logistique ou un autre nombre de voisins pour le kNN.